In [ ]:
#pip installs
#%pip install langchain
#%pip install langchain-ollama
#%pip install langchain-qdrant
#%pip install langchain-qdrant -U
#%pip install pandas
#%pip install fastembed

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 58.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [fastembed]/4 [fastembed]

[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [7]:
# import the main libraries
from langchain_ollama import OllamaEmbeddings
from langchain_qdrant import QdrantVectorStore
from langchain_core.documents import Document
from qdrant_client import QdrantClient, models
from qdrant_client.http.models import Distance, VectorParams
import os
import pandas as pd

# 1. Use Qdrant to create a memory collection and vector store
- collection is named "movie_collection" 
- use COSINE distance

In [ ]:
qdrant_client = QdrantClient(":memory:")
embeddings = OllamaEmbeddings(model="mxbai-embed-large")

# create the collection in Qdrant
qdrant_client.create_collection(
    collection_name="movie_collection",
    vectors_config=VectorParams(size=1024, distance=Distance.COSINE)
)

# create the vector store 
vector_store = QdrantVectorStore(
    client=qdrant_client,
    collection_name="movie_collection", # must be the same as the collection created in Qdrant
    embedding=embeddings
)


# 2. Process “Public Criterion Master List.csv”
- include each field in the metadata, as well as “source: csv”
- “documents” part can include any fields, must at least have title and description

In [16]:
# process the csv
df = pd.read_csv("./dataset/Public Criterion Master List.csv")
csv_list = df.to_dict(orient="records")

csv_docs = [Document(
        page_content="Spine:"+ (str(doc["SPINE"]))+ " " + "Title:"+(str(doc["TITLE (Year)"]))+ \
                    "Director:"+(str(doc["DIRECTOR"]))+ " " + "Country:"+(str(doc["COUNTRY"]))+ " " + \
                    "Description:"+(str(doc["Brief Description"])), 
        metadata={
            "source": "csv", 
            "spine": str(doc["SPINE"]),
            "title_year": str(doc["TITLE (Year)"]),
            "director": str(doc["DIRECTOR"]),
            "country": str(doc["COUNTRY"]),
            "4k_release": str(doc["4K Release?"]),
            "imdb_rating": str(doc["IMDb Rating"]),
            "rotten_tomatoes": str(doc["Rotten Tomatoes"]),
            "count_sort": str(doc["Count (Sort)"]),
            "year": str(doc["YEAR"]),
            "brief_description": str(doc["Brief Description"]),
        }) for doc in csv_list]


# 3. Add the processed data to the vector store with add_documents()

In [ ]:
vector_store.add_documents(
    documents=csv_docs,
    ids=ids
)

[0,
 1,
 2,
 3,
 4,
 5,
 6,
 7,
 8,
 9,
 10,
 11,
 12,
 13,
 14,
 15,
 16,
 17,
 18,
 19,
 20,
 21,
 22,
 23,
 24,
 25,
 26,
 27,
 28,
 29,
 30,
 31,
 32,
 33,
 34,
 35,
 36,
 37,
 38,
 39,
 40,
 41,
 42,
 43,
 44,
 45,
 46,
 47,
 48,
 49,
 50,
 51,
 52,
 53,
 54,
 55,
 56,
 57,
 58,
 59,
 60,
 61,
 62,
 63,
 64,
 65,
 66,
 67,
 68,
 69,
 70,
 71,
 72,
 73,
 74,
 75,
 76]

# 4. Retrieve the point with id 1, using get_by_ids()

In [23]:
vector_store.get_by_ids([1])

[Document(metadata={'source': 'csv', 'spine': '2', 'title_year': 'Seven Samurai (1954)', 'director': 'Akira Kurosawa', 'country': 'Japan', '4k_release': 'True', 'imdb_rating': '8.6/10', 'rotten_tomatoes': '100%', 'count_sort': '2', 'year': '1954.0', 'brief_description': 'A poor village hires seven desperate but experienced samurai to protect them from a ruthless band of marauding bandits.', '_id': 1, '_collection_name': 'movie_collection'}, page_content='Spine:2 Title:Seven Samurai (1954)Director:Akira Kurosawa Country:Japan Description:A poor village hires seven desperate but experienced samurai to protect them from a ruthless band of marauding bandits.')]

# 5. Upsert (add.documents()) "dummy data" to id 1
- use get_by_ids to check

In [24]:
# create a dummy document
dummy_doc = Document(
    page_content="dummy data",
    metadata={"source": "dummy"}
)
# add to the vector store
vector_store.add_documents(documents=[dummy_doc], ids=[1])
# check the added document
vector_store.get_by_ids([1])


[Document(metadata={'source': 'dummy', '_id': 1, '_collection_name': 'movie_collection'}, page_content='dummy data')]

# 6. Delete point id 1

In [25]:
vector_store.delete(ids=[1])

True

# 7. Use add_documents() to restore the original data for id 1

In [27]:
vector_store.add_documents(documents=[csv_docs[1]], ids=[1])

[1]

# 8. Use get_by_ids to check if it has been inserted successfully

In [28]:
vector_store.get_by_ids([1])

[Document(metadata={'source': 'csv', 'spine': '2', 'title_year': 'Seven Samurai (1954)', 'director': 'Akira Kurosawa', 'country': 'Japan', '4k_release': 'True', 'imdb_rating': '8.6/10', 'rotten_tomatoes': '100%', 'count_sort': '2', 'year': '1954.0', 'brief_description': 'A poor village hires seven desperate but experienced samurai to protect them from a ruthless band of marauding bandits.', '_id': 1, '_collection_name': 'movie_collection'}, page_content='Spine:2 Title:Seven Samurai (1954)Director:Akira Kurosawa Country:Japan Description:A poor village hires seven desperate but experienced samurai to protect them from a ruthless band of marauding bandits.')]

# 9. Construct a query statement to get 10 results using similarity_search_with_relevance_scores 
- “Which of these movies deal with war in some way?” Print the results with relevance scores.
- Then, use a score threshold to narrow down the results and print them again